In [ ]:
from collections import Counter, defaultdict
from functools import reduce
from pathlib import Path
from pprint import pprint

ip = get_ipython()
formatter = ip.display_formatter.formatters['text/plain']
formatter.max_seq_length = 10

In [34]:
class Protein:
  def __init__(self, name: str, sequence: str): self.name, self.seq = name, sequence
  def __str__(self): return f"{self.name}: {self.seq}"
  def __repr__(self): return f"{self.name}: {self.seq}"
  def cnt(self, amino_acid: str) -> int: return self.seq.count(amino_acid)
  def seq_len(self) -> int: return len(self.seq)
  def to_counter(self) -> Counter: return Counter(self.seq)

In [35]:
def load_fasta(path: str = "../data/protein.fasta") -> list[Protein]:
  raw = Path(path).read_text().split('>')[1:]
  return [Protein(header.split('|')[2], seq.replace('\n', '')) for entry in raw for header, _, seq in [entry.partition('\n')]]

In [36]:
proteins = load_fasta()
print(f"liczba sekwencji:")
len(proteins)

liczba sekwencji:


4922

In [37]:
print(f"srednia dlugosc sekwencji:")
sum(p.seq_len() for p in proteins)/len(proteins)

srednia dlugosc sekwencji:


147.41284030881755

In [38]:
print(f"liczba leucyn:")
sum(p.cnt('L') for p in proteins)

liczba leucyn:


85410

In [39]:
print(f"liczby amino kwasów:")
sum((p.to_counter() for p in proteins), Counter())

liczby amino kwasów:


Counter({'L': 85410,
         'A': 85319,
         'V': 57632,
         'S': 55177,
         'K': 53712,
         'D': 41955,
         'G': 40134,
         'T': 39170,
         'F': 35807,
         'H': 34298,
         ...})

In [40]:
print("pogrupowane sekwencje:")
groups = reduce(lambda d, p: d[p.seq_len()].append(p.seq) or d, proteins, defaultdict(list))
groups

pogrupowane sekwencje:


defaultdict(list,
            {143: ['MSLSAKEKVIVKDFFAKVSSRSDEIGAEALARLIVVYPQTKSYFAHWKDLSPNGAPVRKHGITVMGGVYEAVSKIDDLAGGLLNLSELHAFVLRVDPVNFKILSHCIMVVLSMLFAEEFTPQIHVAVDKFLALVALALSEKYR',
              'MSLTAKDKKMVKAFWAKVSGKAEDIGCDALSRMLVVYPQTKTYFSHWKDLSPGSAPVRKHGGTIMGGISLAVASIDDISAGLLSLSELHAFQLRIDPANFKILSHNILVVLAVLFPNDFNPEAHVAMDKFLAAVGRALSEKYR',
              'MSLSDKDKAAVKALWAKAEGKAAEMGGEALGRMLVAYPQTKTYFSHWGDLSPQSAKVKKHGAVIMGALGKAVKGIDDLTGTLAALSELHAFKLRVDPANFKILGHNIVLVLAMYFPADFTPEVHLSVEKFLQNVALALSEKYR',
              'MSLSGKDKTVVKAFWDKVAPKAAEIGGEALGRMLTSYPQTKTYFSHWSDLSPESAQVKKHGATIMGAIGDAIGKIDDLQGALAKLSELHAFKLRVDPANFRILAHNIILVLAMYYPADFTPEVHVSVDKFLQNLALALSEKYR',
              'MLTAEDKKLIQQVWGKVGGAEEEIGAETLWRMFHSYPPTKTYFPHFDLSQGSDQIRGHGKKVVAALGTAIKNLDNLSQALSELSNLHAYNLRVDPPCFALQFLAQCLQVVLAVRLGKEYTPEVHSAVDKFMSAVAAVLAEKYR',
              'MLTADDKKLIPQTWEKVVSQCDDAGKDALYRMFVAYPQTKTYFPHFDVGPGSEQVKAHGKKVVAALTNAVKNLDNLSQALSELSNLHAYNLRVDPSCFALQLLAQCIQVVLAVNLGKEYTPEVHSAFDKFLSAVAAVLAEKYR',
           

In [41]:
hamming_mean = lambda seqs: sum(len(seqs)**2 - sum(c**2 for c in Counter(col).values()) for col in zip(*seqs)) / len(seqs)**2
mean_by_group = {seq_len: hamming_mean(seq) for seq_len, seq in groups.items()}
print("srednie odleglosci hamminga:")
{k: round(v, 2) for k, v in mean_by_group.items()}

srednie odleglosci hamminga:


{143: 69.87,
 142: 53.43,
 54: 0.0,
 116: 71.52,
 144: 54.62,
 141: 64.85,
 111: 43.72,
 157: 105.09,
 162: 115.06,
 80: 0.0,
 ...}